# AI agents in STEM — inspect the evidence

Run all cells with **Python (Pyodide)** in the browser, or Python 3.11+ with matplotlib on your computer. First browser launch downloads the runtime/packages. Download your notebook to preserve your work.

These are transparent teaching models. They do not run CalculiX, ORCA, ngspice or GROMACS. The occupancy counts are **synthetic**, not measurements from the lecture. Outcomes connect to Q01–Q08: distinguish execution from acceptance, check units and assumptions, and keep uncertainty visible.


In [ ]:
"""Small teaching models in SI units. No ORCA, ngspice or MD solver is run here."""
import math

def axial_bar(load=1000.0, length=0.01, area=4e-6, modulus=210e9):
    """Uniform uniaxial small-strain bar: fixed left, axial force at right.

    Neglect lateral restraint, bending, plasticity and thermal effects.
    SI: force N, length m, area m², modulus Pa; returns extension m.
    """
    if not all(math.isfinite(v) for v in (load,length,area,modulus)) or min(length,area,modulus)<=0:
        raise ValueError('Finite load and positive length, area and modulus required')
    return load*length/(area*modulus)

def bar_discrete_solution(load=1000.0,length=0.01,area=4e-6,modulus=210e9,elements=4):
    """Independent assembled 1D linear finite-element solve by elimination.

    Uniform bar with one end fixed. Exact nodal agreement for this patch is
    expected and is not a convergence study for arbitrary 3D geometries.
    """
    axial_bar(load,length,area,modulus)
    if not isinstance(elements,int) or not 1<=elements<=100:raise ValueError('1..100 elements required')
    n=elements;k=modulus*area/(length/n)
    K=[[0.0]*n for _ in range(n)];f=[0.0]*n;f[-1]=load
    for e in range(n):
        for a,signa in [(e,-1),(e+1,1)]:
            for b,signb in [(e,-1),(e+1,1)]:
                if a and b:K[a-1][b-1]+=k*signa*signb
    for col in range(n):
        pivot=K[col][col]
        for row in range(col+1,n):
            factor=K[row][col]/pivot
            for j in range(col,n):K[row][j]-=factor*K[col][j]
            f[row]-=factor*f[col]
    u=[0.0]*n
    for i in range(n-1,-1,-1):u[i]=(f[i]-sum(K[i][j]*u[j]for j in range(i+1,n)))/K[i][i]
    return [0.0]+u

def amplifier(frequency,gain=10.0,pole_hz=100010.0):
    """Illustrative small-signal, single-pole model, H=A0/(1+j f/fp)."""
    if not all(math.isfinite(x) for x in (frequency,gain,pole_hz)) or frequency<0 or gain<=0 or pole_hz<=0:raise ValueError('Invalid amplifier parameter')
    return gain/complex(1,frequency/pole_hz)

def cutoff_bisection(gain=10.0,pole_hz=100010.0):
    """Locate half-power magnitude numerically; do not return pole_hz directly."""
    target=abs(amplifier(0,gain,pole_hz))/math.sqrt(2)
    lo,hi=0.0,100*pole_hz
    for _ in range(80):
        mid=(lo+hi)/2
        if abs(amplifier(mid,gain,pole_hz))>target:lo=mid
        else:hi=mid
    return (lo+hi)/2

def occupancy_summary(counts):
    """Counts for a stated frame/region definition; no MD trajectory analysis."""
    if not counts or any(not isinstance(x,int) or x<0 for x in counts):raise ValueError('Nonnegative integer counts required')
    return {'frames':len(counts),'mean':sum(counts)/len(counts),'nonzero_fraction':sum(x>0 for x in counts)/len(counts),'maximum':max(counts)}

def validate():
    for n in [1,2,4,8,16]:
        for force in [-500,0,1000,2000]:
            u=bar_discrete_solution(load=force,elements=n)
            exact=axial_bar(load=force)
            assert max(abs(x-exact*i/n)for i,x in enumerate(u))<1e-15
    for gain,pole in [(10,100010),(8,200000),(3,1)]:
        fc=cutoff_bisection(gain,pole)
        assert abs(fc/pole-1)<1e-12
        for f in [0,pole,10*pole]:assert abs(abs(amplifier(f,gain,pole))-gain/math.sqrt(1+(f/pole)**2))<1e-12
    synthetic=[0]*18+[1,1]
    water=occupancy_summary(synthetic)
    assert water['mean']==0.1 and water['nonzero_fraction']==0.1 and water['maximum']==1
    return {'status':'pass','bar_extension_m':axial_bar(),'bar_force_N':1000,'bar_elements_tested':[1,2,4,8,16],'amplifier_cutoff_hz':cutoff_bisection(),'bandwidth_requirement_hz':150000,'bandwidth_pass':cutoff_bisection()>=150000,'synthetic_occupancy':water,'limits':'1D small-strain uniform bar, single-pole small-signal circuit, synthetic occupancy; not validation of the recorded external solvers or real bilayer.'}

if __name__=='__main__':
    import json
    print(json.dumps(validate(),indent=2))


In [ ]:
checks=validate()
print("SCIENTIFIC_CHECKS", checks)
assert checks["status"]=="pass"


## 1. Axial bar — units before confidence (Q02–Q04)

Model: uniform linear elastic bar; fixed left end, axial force at the right; no bending or thermal effects. The independent discrete system assembles local stiffness matrices and solves equilibrium. Exact nodal agreement for this simple patch does not validate arbitrary 3D meshes.

Predict: if area doubles at fixed force, what happens to extension? Change `area_m2` and test your prediction. Explain why entering 210 instead of 210e9 for a modulus in Pa would be wrong.


In [ ]:
import matplotlib.pyplot as plt
length_m=0.01
area_m2=4e-6
force_N=1000.0
E_Pa=210e9
u=bar_discrete_solution(force_N,length_m,area_m2,E_Pa,8)
x=[length_m*i/8 for i in range(9)]
exact=[axial_bar(force_N,t,area_m2,E_Pa) if t else 0 for t in x]
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot([t*1000 for t in x],[t*1e6 for t in exact],label="Analytical")
ax.plot([t*1000 for t in x],[t*1e6 for t in u],"o",label="Assembled 1D elements")
ax.set(xlabel="Position (mm)",ylabel="Axial displacement (µm)",title="Uniform bar: independent equilibrium check")
ax.legend();ax.grid(alpha=.25);plt.show()
print("BAR_END_UM",u[-1]*1e6)
assert abs(axial_bar(force_N,length_m,2*area_m2,E_Pa)/u[-1]-.5)<1e-12


## 2. Two requirements, two checks (Q06 and Q06A)

Small-signal, single-pole model: `H(f)=A0/(1+j f/fp)`. Here A0=10 and fp=100010 Hz. The cutoff is relative to the low-frequency gain, at half power (about −3.0103 dB). This model omits slew rate, clipping, noise, component variation, layout parasitics and stability of a real circuit.

Predict the acceptance result before running. A circuit must meet both gain ≥10 and cutoff ≥150 kHz. Then try gain=8 and pole=200000 Hz — the transfer task from the course.


In [ ]:
gain=10.0
pole=100010.0
frequencies=[10**(2+i/40) for i in range(181)]
db=[20*math.log10(abs(amplifier(f,gain,pole))) for f in frequencies]
fc=cutoff_bisection(gain,pole)
fig,ax=plt.subplots(figsize=(7,3.5))
ax.semilogx(frequencies,db,label="Model AC magnitude")
ax.axvline(fc,color="darkorange",label=f"Cutoff {fc/1000:.2f} kHz")
ax.axvline(150000,color="darkred",linestyle="--",label="Required minimum cutoff")
ax.set(xlabel="Frequency (Hz)",ylabel="Gain (dB)",title="Gain alone is only half the test")
ax.grid(alpha=.25);ax.legend();plt.show()
print("CIRCUIT_ACCEPTANCE",{"gain_pass":gain>=10,"bandwidth_pass":fc>=150000,"accepted":gain>=10 and fc>=150000})
print("TRANSFER",{"gain_pass":8>=10,"bandwidth_pass":cutoff_bisection(8,200000)>=150000})
assert not (8>=10 and cutoff_bisection(8,200000)>=150000)


## 3. A mean is not a zero-event claim (Q08)

Definition: each synthetic count is the number of water molecules meeting one hypothetical local distance criterion in one frame. It is not a deep-bilayer occupancy definition. The sequence has clustered events; treating these frames as independent samples would understate sampling uncertainty.

Calculate the nonzero-frame fraction and the maximum as well as the mean. Move the two events apart without changing the mean: which temporal conclusions change? A real analysis needs region definitions, equilibration checks, correlation times and independent sampling; this tiny example estimates no real confidence interval.


In [ ]:
counts=[0]*18+[1,1]
stats=occupancy_summary(counts)
fig,axes=plt.subplots(1,2,figsize=(8,3))
axes[0].step(range(len(counts)),counts,where="mid")
axes[0].set(xlabel="Synthetic frame",ylabel="Local water count",title="Intermittent occupancy")
axes[1].bar(["zero","nonzero"],[sum(x==0 for x in counts),sum(x>0 for x in counts)])
axes[1].set(ylabel="Frames",title="Distribution matters")
fig.tight_layout();plt.show()
print("SYNTHETIC_OCCUPANCY",stats)
assert stats["mean"]==.1 and stats["maximum"]==1
print("LAB_COMPLETE: 3 figures; independent bar/circuit checks; synthetic occupancy; transfer task")


## 4. Write the handoff (Q01, Q03, Q05, Q05A)

For one workflow in your own field, write: the question; inputs and units; physical and numerical assumptions; software/model versions; what actually executed; an independent acceptance check; uncertainty and remaining limitations; and the exact artifacts another researcher needs.

An ORCA submission ID establishes scheduling acceptance, not convergence or chemical accuracy. An AI-selected model remains an assumption to examine. Do not claim the simple models above validate the recorded production simulations.
